In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from pandas.api.types import is_datetime64_any_dtype

In [2]:
model = joblib.load("model_retard_rf.joblib")
features_train = joblib.load("features_train.joblib")

In [3]:
df_new = pd.read_csv("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/processed/test_philippe/SignofFlightsDataset_72h_20260409_023452_CLEAN.csv")
df_new.head()

,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,LABEL_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination,retard arrivée
0,DATE_GENERATION,2026-04-09 02:34:54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-08,2026-04-08,EC 4532,EasyJet Europe,NCE,CDG,2,2D,2026-04-08 21:55+02:00,2026-04-08 23:35+02:00,...,NaN,2.0,0.0,2.0,0.0,NaN,NaN,NaN,0.0,0.0
2,2026-04-08,2026-04-08,U2 4532,easyJet,NCE,CDG,2,2D,2026-04-08 21:55+02:00,2026-04-08 23:35+02:00,...,NaN,2.0,0.0,2.0,0.0,NaN,NaN,NaN,0.0,0.0
3,2026-04-09,2026-04-09,AF 7300,Air France,CDG,NCE,2F,2,2026-04-09 06:45+02:00,2026-04-09 08:15+02:00,...,NaN,2473.0,2502.0,4975.0,0.0,173.0,174.0,347.0,1.0,0.0
4,2026-04-09,2026-04-09,AF 7301,Air France,NCE,CDG,2,2F,2026-04-09 09:05+02:00,2026-04-09 10:40+02:00,...,NaN,173.0,174.0,347.0,1.0,2473.0,2502.0,4975.0,0.0,0.0


In [4]:
df_new.shape

(1819, 95)

In [5]:
print(df_new.iloc[0])
df_new = df_new.iloc[1:].reset_index(drop=True)
print("First line suppressed")

# df_new = df_new.sample(500, random_state=42).copy() # code for sampling if we got a bigger data

flight_date                             DATE_GENERATION
movement_date                       2026-04-09 02:34:54
flight_number                                       NaN
airline                                             NaN
airport_origin                                      NaN
                                           ...         
nombre_departs_destination                          NaN
nombre_arrivees_destination                         NaN
somme_depart_arrivee_destination                    NaN
congestion_destination                              NaN
retard arrivée                                      NaN
Name: 0, Length: 95, dtype: object
First line suppressed


In [6]:
cols_to_drop = [
    'cloud_base_dep', 'visibility_dep', 'cloud_base_arr', 'visibility_arr',
    'Vacances Scolaires', 'Label des Vacances', 'Label Jour Ferié',
    'LABEL_LYON', 'LABEL_TOULOUSE', 'LABEL_NICE', 'LABEL_MARSEILLE',
    'LABEL_CDG', 'LABEL_ORLY',
    "actual_arrival", "actual_departure", "actual_source_arrival",
    "actual_source_departure", "arrival_advance_min", "departure_delay_min",
    "departure_advance_min", "estimated_departure", "estimated_arrival",
    "arrival_delay_min"
]

df_new = df_new.drop(columns=cols_to_drop, errors="ignore")

In [7]:
target_col = "retard arrivée"

if target_col in df_new.columns:
    y_true = df_new[target_col].astype(int)
    X_new = df_new.drop(columns=[target_col], errors="ignore")
else:
    y_true = None
    X_new = df_new.copy()

In [8]:
datetime_cols = [
    "flight_date",
    "movement_date",
    "scheduled_departure",
    "scheduled_arrival",
    "time_dep",
    "time_arr"
]

def add_datetime_features_safe(df, datetime_cols):
    df = df.copy()
    bad_datetime_cols = []

    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

            if not is_datetime64_any_dtype(df[col]):
                bad_datetime_cols.append(col)

    usable_datetime_cols = [col for col in datetime_cols if col in df.columns and col not in bad_datetime_cols]

    if "flight_date" in usable_datetime_cols:
        df["flight_month"] = df["flight_date"].dt.month
        df["flight_day"] = df["flight_date"].dt.day
        df["flight_dayofweek"] = df["flight_date"].dt.dayofweek

    if "scheduled_departure" in usable_datetime_cols:
        df["sched_dep_hour"] = df["scheduled_departure"].dt.hour
        df["sched_dep_minute"] = df["scheduled_departure"].dt.minute

    if "scheduled_arrival" in usable_datetime_cols:
        df["sched_arr_hour"] = df["scheduled_arrival"].dt.hour
        df["sched_arr_minute"] = df["scheduled_arrival"].dt.minute

    print("Colonnes datetime problématiques droppées :", bad_datetime_cols)

    df = df.drop(columns=datetime_cols, errors="ignore")

    return df

X_new = add_datetime_features_safe(X_new, datetime_cols)

Colonnes datetime problématiques droppées : []


In [9]:
# Ajouter colonnes manquantes
for col in features_train:
    if col not in X_new.columns:
        X_new[col] = 0

# Garder uniquement les colonnes du train dans le bon ordre
X_new = X_new[features_train]

In [10]:
y_pred = model.predict(X_new)
y_proba = model.predict_proba(X_new)[:, 1]

In [11]:
df_result = df_new.copy()

df_result.insert(0, "prediction", y_pred)
df_result.insert(1, "proba_retard", y_proba)

In [12]:
df_result.to_csv("predictions_retard_vols.csv", index=False)
print("CSV exporté")

CSV exporté
